In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import StringType
from pyspark.sql import Window
import pandas as pd
import numpy as np

In [3]:
spark

In [4]:
df = spark.table('production.dwh.fact_booking')

In [5]:
df.printSchema()

root
 |-- date_of_checkout: timestamp (nullable = true)
 |-- booking_id: long (nullable = true)
 |-- shopping_cart_id: long (nullable = true)
 |-- tour_id: long (nullable = true)
 |-- tour_option_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- status_id: integer (nullable = true)
 |-- master_language_id: long (nullable = true)
 |-- customer_id_anon: string (nullable = true)
 |-- customer_address_id_anon: string (nullable = true)
 |-- customer_address_lead_traveller_id_anon: string (nullable = true)
 |-- customer_country_id: long (nullable = true)
 |-- customer_language_id: long (nullable = true)
 |-- customer_currency_id: long (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- token: string (nullable = true)
 |-- hash_code: string (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- supplier_currency_id: long (nullable = true)
 |-- reseller_id: long (nul

In [6]:
print(f"Row count: {df.count():,}")

Row count: 132,193,293


In [ ]:
print(f"Row count: {df.count():,}")

## Top 10 selling tours in 2024 and their tour options

In [23]:
top_tours_sql = """
WITH top_tours AS (
    SELECT
        tour_id,
        COUNT(booking_id)  AS booking_count,
        SUM(tickets)       AS total_tickets,
        ROUND(SUM(gmv), 2) AS total_gmv_eur
    FROM production.dwh.fact_booking
    WHERE
        YEAR(date_of_checkout) = 2024
        AND is_fraud = false
        AND date_of_cancelation IS NULL
    GROUP BY tour_id
    ORDER BY booking_count DESC
    LIMIT 10
)

SELECT
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
FROM top_tours t
JOIN production.dwh.fact_booking b
    ON  b.tour_id = t.tour_id
    AND YEAR(b.date_of_checkout) = 2024
    AND b.is_fraud = false
    AND b.date_of_cancelation IS NULL
GROUP BY
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
ORDER BY t.booking_count DESC, t.tour_id, b.tour_option_id
"""

df_top_tours = spark.sql(top_tours_sql)

In [24]:
display(df_top_tours)

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737
3,50027,121163,311630,10416319.95,654640
4,50027,121163,311630,10416319.95,654660
5,50027,121163,311630,10416319.95,741729
6,53791,107484,271705,3435836.97,74290
7,193940,102208,278280,4338708.96,300869
8,73250,98289,249158,3220068.00,376779
9,145779,81545,172828,4345691.44,218301


In [25]:
display(df_top_tours.limit(10))

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737
3,50027,121163,311630,10416319.95,654640
4,50027,121163,311630,10416319.95,654660
5,50027,121163,311630,10416319.95,741729
6,53791,107484,271705,3435836.97,74290
7,193940,102208,278280,4338708.96,300869
8,73250,98289,249158,3220068.00,376779
9,145779,81545,172828,4345691.44,218301


In [55]:
display(df_top_tours.limit(5))

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737
3,50027,121163,311630,10416319.95,654640
4,50027,121163,311630,10416319.95,654660


In [56]:
df_top_tours.count()

47

In [57]:
dim_option = spark.table('production.dwh.dim_tour_option').select('tour_option_id', 'tour_option_title')

df_top_tours_named = df_top_tours.join(dim_option, on='tour_option_id', how='left') \
    .select('tour_id', 'tour_option_id', 'tour_option_title', 'booking_count', 'total_tickets', 'total_gmv_eur') \
    .orderBy('booking_count', ascending=False)

display(df_top_tours_named)

,tour_id,tour_option_id,tour_option_title,booking_count,total_tickets,total_gmv_eur
0,62214,963208,Vatican: Museums & Sistine Chapel Entrance Ticket,128680,362286,11852559.98
1,62214,963214,Ticket with Audioguide,128680,362286,11852559.98
2,62214,1037737,2024 Ticket Only,128680,362286,11852559.98
3,50027,741729,Entry Ticket with Audio Guide (no Tower Access),121163,311630,10416319.95
4,50027,654640,Ticket with Passion Facade Tower Access & Audio Guide App,121163,311630,10416319.95
5,50027,654660,Ticket with Nativity Facade Tower Access & Audio Guide App,121163,311630,10416319.95
6,53791,74290,Barcelona: Park Güell Admission Ticket,107484,271705,3435836.97
7,193940,300869,Paris: 1-Hour Seine Cruise departing from the Eiffel Tower,102208,278280,4338708.96
8,73250,376779,Budapest: City Highlights Evening Cruise with Welcome Drink,98289,249158,3220068.00
9,145779,218301,Paris: Louvre Museum Timed-Entrance Ticket,81545,172828,4345691.44


In [7]:
# Tours qualifying for forecasting:
#   - window: [end_date - 3y6m, end_date]  (3y6m = 42 months)
#   - history must span >= 3 years within the window
#   - avg active tickets per day > 10  (denominator = 3*365, all days in window)
END_DATE = "2026-05-01"

# Compute start date dynamically via Spark SQL (ADD_MONTHS handles month-end edge cases)
START_DATE = spark.sql(f"SELECT ADD_MONTHS(DATE('{END_DATE}'), -42)").collect()[0][0]
print(f"Window: {START_DATE} → {END_DATE}")

df_forecast_tours = spark.sql(f"""
    SELECT
        tour_id,
        MIN(DATE(date_of_travel))                                AS first_travel_date,
        MAX(DATE(date_of_travel))                                AS last_travel_date,
        DATEDIFF(
            MAX(DATE(date_of_travel)),
            MIN(DATE(date_of_travel))
        )                                                        AS history_days,
        SUM(tickets)                                             AS total_tickets,
        ROUND(SUM(tickets) / (3 * 365), 2)                      AS avg_tickets_per_day
    FROM production.dwh.fact_booking
    WHERE status_id = 1
      AND DATE(date_of_travel) BETWEEN '{START_DATE}' AND '{END_DATE}'
    GROUP BY tour_id
    HAVING DATEDIFF(
               MAX(DATE(date_of_travel)),
               MIN(DATE(date_of_travel))
           ) >= 3 * 365
       AND SUM(tickets) / (3 * 365) > 10
    ORDER BY avg_tickets_per_day DESC
""")

Window: 2022-11-01 → 2026-05-01


In [8]:
print(f"Tours qualifying for forecasting: {df_forecast_tours.count():,}")
display(df_forecast_tours.limit(5))

Tours qualifying for forecasting: 2,160


,tour_id,first_travel_date,last_travel_date,history_days,total_tickets,avg_tickets_per_day
0,62214,2022-11-02,2026-04-30,1275,1442128,1317.01
1,193940,2022-11-01,2026-05-01,1277,1118722,1021.66
2,50027,2022-11-01,2026-05-01,1277,1080836,987.06
3,53791,2022-11-01,2026-05-01,1277,930741,849.99
4,416276,2022-11-01,2026-05-01,1277,710422,648.79


In [9]:
# Daily time series: total tickets sold per tour (all options summed)
# Filtered to the same window and only tours selected in df_forecast_tours
df_timeseries = (
    spark.table('production.dwh.fact_booking')
    .filter(
        (F.col('status_id') == 1) &
        (F.col('date_of_travel').isNotNull()) &
        (F.col('date_of_travel').cast('date') >= F.lit(str(START_DATE))) &
        (F.col('date_of_travel').cast('date') <= F.lit(END_DATE))
    )
    .join(df_forecast_tours.select('tour_id'), on='tour_id', how='inner')
    .groupBy('tour_id', F.col('date_of_travel').cast('date').alias('date'))
    .agg(F.sum('tickets').alias('tickets_sold'))
    .orderBy('tour_id', 'date')
)

print(f"Total rows: {df_timeseries.count():,}")
df_timeseries.show(20)

Total rows: 2,204,656


+-------+----------+------------+
|tour_id|      date|tickets_sold|
+-------+----------+------------+
|    521|2022-11-01|           7|
|    521|2022-11-08|           1|
|    521|2022-11-10|           3|
|    521|2022-11-13|           4|
|    521|2022-11-15|           3|
|    521|2022-11-17|           1|
|    521|2022-11-20|           2|
|    521|2022-11-22|          12|
|    521|2022-11-24|          11|
|    521|2022-11-27|           1|
|    521|2022-11-29|           9|
|    521|2022-12-01|           2|
|    521|2022-12-04|           5|
|    521|2022-12-06|           2|
|    521|2022-12-08|           2|
|    521|2022-12-11|           7|
|    521|2022-12-15|           2|
|    521|2022-12-18|           9|
|    521|2022-12-20|           6|
|    521|2022-12-27|          18|
+-------+----------+------------+
only showing top 20 rows



In [13]:
df_timeseries.toPandas().to_csv('data.csv', header=True, index=False)

In [10]:
import plotly.graph_objects as go

TOUR_ID = 63242

pdf = (
    df_timeseries.filter(F.col('tour_id') == TOUR_ID)
    .toPandas()
    .sort_values('date')
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=pdf['date'], y=pdf['tickets_sold'], mode='lines', name='tickets sold'))
fig.update_layout(
    title=f'Daily tickets sold — tour {TOUR_ID}',
    xaxis_title='Date',
    yaxis_title='Tickets sold',
    hovermode='x unified',
)
;
# fig.show()

''

In [11]:
fig

In [54]:
pdf

,tour_id,date,tickets_sold
0,63242,2022-11-01,8
1,63242,2022-11-02,17
2,63242,2022-11-03,12
3,63242,2022-11-04,25
4,63242,2022-11-05,18
...,...,...,...
1166,63242,2026-04-27,56
1167,63242,2026-04-28,63
1168,63242,2026-04-29,55
1169,63242,2026-04-30,75


In [58]:
START_DATE

datetime.date(2022, 11, 1)

In [60]:
full_idx = pd.date_range(start=str(START_DATE), end=END_DATE, freq='D')

pdf_full = (
    pdf[['tickets_sold']]
    .set_index(pd.to_datetime(pdf['date']))
    .reindex(full_idx)
    .fillna({'tickets_sold': 0})
    .astype({'tickets_sold': int})
    .reset_index()
    .rename(columns={'index': 'date'})
)
pdf_full['date'] = pdf_full['date'].dt.date
pdf_full.insert(0, 'tour_id', TOUR_ID)

print(f"Total days    : {len(pdf_full):,}")
print(f"Missing filled: {(pdf_full['tickets_sold'] == 0).sum():,}")
pdf_full

Total days    : 1,278
Missing filled: 107


,tour_id,date,tickets_sold
0,63242,2022-11-01,8
1,63242,2022-11-02,17
2,63242,2022-11-03,12
3,63242,2022-11-04,25
4,63242,2022-11-05,18
...,...,...,...
1273,63242,2026-04-27,56
1274,63242,2026-04-28,63
1275,63242,2026-04-29,55
1276,63242,2026-04-30,75


In [62]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pdf_full['date'], y=pdf_full['tickets_sold'], mode='lines', name='tickets sold'))
fig.update_layout(
    title=f'Daily tickets sold — tour {TOUR_ID}',
    xaxis_title='Date',
    yaxis_title='Tickets sold',
    hovermode='x unified',
)
;

''

In [63]:
fig

In [65]:
full_idx = pd.date_range(start=str(START_DATE), end=END_DATE, freq='D')

pdf_full = (
    pdf[['tickets_sold']]
    .set_index(pd.to_datetime(pdf['date']))
    .reindex(full_idx)
    .assign(is_missing_date=lambda x: x['tickets_sold'].isna())
    .fillna({'tickets_sold': 0})
    .astype({'tickets_sold': int})
    .reset_index()
    .rename(columns={'index': 'date'})
)
pdf_full['date'] = pdf_full['date'].dt.date
pdf_full.insert(0, 'tour_id', TOUR_ID)
pdf_full = pdf_full[['tour_id', 'date', 'tickets_sold', 'is_missing_date']]

print(f"Total days    : {len(pdf_full):,}")
print(f"Missing filled: {pdf_full['is_missing_date'].sum():,}")
pdf_full

Total days    : 1,278
Missing filled: 107


,tour_id,date,tickets_sold,is_missing_date
0,63242,2022-11-01,8,False
1,63242,2022-11-02,17,False
2,63242,2022-11-03,12,False
3,63242,2022-11-04,25,False
4,63242,2022-11-05,18,False
...,...,...,...,...
1273,63242,2026-04-27,56,False
1274,63242,2026-04-28,63,False
1275,63242,2026-04-29,55,False
1276,63242,2026-04-30,75,False


In [66]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pdf_full['date'], y=pdf_full['tickets_sold'], mode='lines', name='tickets sold'))
fig.update_layout(
    title=f'Daily tickets sold — tour {TOUR_ID}',
    xaxis_title='Date',
    yaxis_title='Tickets sold',
    hovermode='x unified',
)
;

''

In [67]:
fig

In [69]:
display(pdf_full[pdf_full['is_missing_date'] == True])

,tour_id,date,tickets_sold,is_missing_date
41,63242,2022-12-12,0,True
42,63242,2022-12-13,0,True
43,63242,2022-12-14,0,True
44,63242,2022-12-15,0,True
45,63242,2022-12-16,0,True
...,...,...,...,...
1190,63242,2026-02-03,0,True
1191,63242,2026-02-04,0,True
1192,63242,2026-02-05,0,True
1193,63242,2026-02-06,0,True


In [71]:
from IPython.display import display as display2
display2(pdf_full)

,tour_id,date,tickets_sold,is_missing_date
0,63242,2022-11-01,8,False
1,63242,2022-11-02,17,False
2,63242,2022-11-03,12,False
3,63242,2022-11-04,25,False
4,63242,2022-11-05,18,False
...,...,...,...,...
1273,63242,2026-04-27,56,False
1274,63242,2026-04-28,63,False
1275,63242,2026-04-29,55,False
1276,63242,2026-04-30,75,False


In [72]:
from itables import show
show(pdf_full)

Loading ITables v2.8.0 from the internet... (need help?)


In [76]:
# pdf_full.style.set_properties(**{'overflow-y': 'scroll', 'display': 'block', 'height': '20px'})

In [80]:
pdf_full[:50].style.set_properties(**{'overflow-y': 'scroll', 'height': '2px'})

,tour_id,date,tickets_sold,is_missing_date
0,63242,2022-11-01,8,False
1,63242,2022-11-02,17,False
2,63242,2022-11-03,12,False
3,63242,2022-11-04,25,False
4,63242,2022-11-05,18,False
5,63242,2022-11-06,7,False
6,63242,2022-11-07,19,False
7,63242,2022-11-08,8,False
8,63242,2022-11-09,13,False
9,63242,2022-11-10,18,False


In [120]:
import datetime

TEST_START = datetime.date(2026, 2, 1)
TEST_END   = datetime.date(2026, 5, 1)
TRAIN_START = datetime.date(TEST_START.year - 3, TEST_START.month, TEST_START.day)
TRAIN_END   = TEST_START - datetime.timedelta(days=1)

df_train = pdf_full[(pdf_full['date'] >= TRAIN_START) & (pdf_full['date'] <= TRAIN_END)].copy()
df_test  = pdf_full[(pdf_full['date'] >= TEST_START)  & (pdf_full['date'] <= TEST_END)].copy()

print(f"Train: {TRAIN_START} → {TRAIN_END}  ({len(df_train):,} days)")
print(f"Test : {TEST_START}  → {TEST_END}   ({len(df_test):,} days)")

Train: 2023-02-01 → 2026-01-31  (1,096 days)
Test : 2026-02-01  → 2026-05-01   (90 days)


In [121]:
df_train

,tour_id,date,tickets_sold,is_missing_date
92,63242,2023-02-01,0,True
93,63242,2023-02-02,0,True
94,63242,2023-02-03,0,True
95,63242,2023-02-04,0,True
96,63242,2023-02-05,0,True
...,...,...,...,...
1183,63242,2026-01-27,15,False
1184,63242,2026-01-28,4,False
1185,63242,2026-01-29,3,False
1186,63242,2026-01-30,17,False


In [122]:
df_test_lookup = df_test[['date']].copy()
df_test_lookup['date_prev_year'] = (pd.to_datetime(df_test_lookup['date']) - pd.DateOffset(years=1)).dt.date

df_test_eval1 = (
    df_test_lookup
    .merge(pdf_full[['date', 'tickets_sold']].rename(columns={'date': 'date_prev_year', 'tickets_sold': 'forecast1'}), on='date_prev_year', how='left')
    .merge(df_test, on='date', how='left')
    [['tour_id', 'date', 'date_prev_year', 'tickets_sold', 'is_missing_date', 'forecast1']]
)

print(f"Missing forecasts (no same date last year): {df_test_eval1['forecast1'].isna().sum()}")
df_test_eval

Missing forecasts (no same date last year): 0


,tour_id,date,date_prev_year,tickets_sold,is_missing_date,forecast1
0,63242,2026-02-01,2025-02-01,19,False,5
1,63242,2026-02-02,2025-02-02,0,True,14
2,63242,2026-02-03,2025-02-03,0,True,5
3,63242,2026-02-04,2025-02-04,0,True,6
4,63242,2026-02-05,2025-02-05,0,True,6
...,...,...,...,...,...,...
55,63242,2026-03-28,2025-03-28,59,False,51
56,63242,2026-03-29,2025-03-29,61,False,52
57,63242,2026-03-30,2025-03-30,61,False,49
58,63242,2026-03-31,2025-03-31,54,False,52


In [123]:
def nearest_same_weekday_prev_year(d):
    ref = (pd.Timestamp(d) - pd.DateOffset(years=1)).date()
    delta = (ref.weekday() - d.weekday()) % 7   # days to subtract from ref
    if delta > 3:
        delta -= 7                               # closer to go forward instead
    return ref - datetime.timedelta(days=delta)

df_test_lookup = df_test[['date']].copy()
df_test_lookup['date_prev_year'] = df_test_lookup['date'].apply(nearest_same_weekday_prev_year)

df_test_eval2 = (
    df_test_lookup
    .merge(pdf_full[['date', 'tickets_sold']].rename(columns={'date': 'date_prev_year', 'tickets_sold': 'forecast2'}), on='date_prev_year', how='left')
    .merge(df_test, on='date', how='left')
    [['tour_id', 'date', 'date_prev_year', 'tickets_sold', 'is_missing_date', 'forecast2']]
)

print(f"Missing forecasts (no same date last year): {df_test_eval2['forecast2'].isna().sum()}")
df_test_eval2

Missing forecasts (no same date last year): 0


,tour_id,date,date_prev_year,tickets_sold,is_missing_date,forecast2
0,63242,2026-02-01,2025-02-02,19,False,14
1,63242,2026-02-02,2025-02-03,0,True,5
2,63242,2026-02-03,2025-02-04,0,True,6
3,63242,2026-02-04,2025-02-05,0,True,6
4,63242,2026-02-05,2025-02-06,0,True,12
...,...,...,...,...,...,...
85,63242,2026-04-27,2025-04-28,56,False,47
86,63242,2026-04-28,2025-04-29,63,False,41
87,63242,2026-04-29,2025-04-30,55,False,42
88,63242,2026-04-30,2025-05-01,75,False,43


In [124]:
wape = df_test_eval1['tickets_sold'].sub(df_test_eval1['forecast1']).abs().sum() / df_test_eval1['tickets_sold'].sum()
print(f"WAPE: {wape:.2%}")

WAPE: 38.94%


In [125]:
wape = df_test_eval2['tickets_sold'].sub(df_test_eval2['forecast2']).abs().sum() / df_test_eval2['tickets_sold'].sum()
print(f"WAPE: {wape:.2%}")

WAPE: 36.50%


In [126]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_test_eval2['date'], y=df_test_eval2['tickets_sold'], mode='lines', name='tickets sold'))
fig.add_trace(go.Scatter(x=df_test_eval2['date'], y=df_test_eval2['forecast2'], mode='lines', name='tickets sold'))
fig.update_layout(
    title=f'Daily tickets sold — tour {TOUR_ID}',
    xaxis_title='Date',
    yaxis_title='Tickets sold',
    hovermode='x unified',
)
;

''

In [127]:
fig